# Retrieval v2 — cấu hình chung và graph nguồn

Runtime dùng một policy cho mọi Chương đã nạp. Lịch sử notebook C1–C15/M0–M5 được giữ tại `artifacts/experiments/run_generic_graph_v2/baseline/notebooks/02_test_retrieval.ipynb`; các output đó không đại diện runtime v2.

Benchmark là hồi quy trên 940 ca đã xem, không là holdout mù. B0: hierarchy + hybrid; B1: thêm graph context search; B1 là mặc định; Notes có liên kết nguồn được trả riêng sau xếp hạng. B2 đưa Notes vào reranker là thí nghiệm tùy chọn. Không tune theo chương.


In [ ]:
from pathlib import Path
import sys
import json
import subprocess

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config/hybrid_retrieval.json").exists()
RUN_ID = "run_generic_graph_v2_budget"
policy = json.loads((ROOT / "config/hybrid_retrieval.json").read_text())
display(policy)


## 1. Chuẩn bị manifest và chấm B0/B1 trên toàn bộ ca

`--resume` kiểm tra snapshot, model, query, policy và code hash; khi thay nguồn/code cần dùng RUN_ID mới. Nhãn chỉ được dùng để chấm điểm, không dùng dựng graph hay truy xuất.


In [ ]:
subprocess.run([sys.executable, "scripts/run_retrieval_experiment.py", "prepare", "--run-id", RUN_ID], cwd=ROOT, check=True)
for method in ("B0", "B1"):
    subprocess.run([sys.executable, "scripts/run_retrieval_experiment.py", "run", "--method", method, "--run-id", RUN_ID, "--resume"], cwd=ROOT, check=True)


In [ ]:
import pandas as pd

folder = ROOT / "artifacts/experiments" / RUN_ID
results = {method: json.loads((folder / f"{method}_metrics.json").read_text()) for method in ("B0", "B1")}
display(pd.DataFrame({method: result["ALL"] for method, result in results.items()}).T)
display(pd.DataFrame(results[policy["default_method"]]["sections"]).T)
display(pd.DataFrame(results[policy["default_method"]]["chapters"]).T)


## 2. QA hồi quy và latency khi bỏ cache

Các lệnh dưới dùng policy đã khóa. Latency warm-model/cold-query khác với thời gian replay ablation và chưa là benchmark tải đồng thời.


In [ ]:
subprocess.run([sys.executable, "scripts/run_runtime_regression.py", "--run-id", RUN_ID], cwd=ROOT, check=True)
subprocess.run([sys.executable, "scripts/measure_fresh_runtime_latency.py", "--run-id", RUN_ID], cwd=ROOT, check=True)

subprocess.run([sys.executable, "scripts/finalize_retrieval_report.py", "--run-id", RUN_ID], cwd=ROOT, check=True)

## 3. Kiểm chứng mở rộng kỹ thuật

`RAG_RUN_DB_TESTS=1 .venv/bin/pytest tests/` kiểm tra nguồn ngoài ba Phần qua adapter/ingest/routing/context/reranker. Fixture dùng model doubles để kiểm tra đường xử lý; không công bố accuracy cho Chương 72 từ fixture này. Thêm chương thật: đăng ký files/tariffs trong source_manifest, audit → build → embed → ingest rồi đo chất lượng.
